# Laguna XS.2 v20.1: Corrected Geodesic SFT with Proper A-Freeze + Norm-Matching

**What changed from v20**:
1. **A is FROZEN** (`requires_grad=False`) — geodesic constraint actually enforced
2. **Norm-matched** — `A₀` scaled to Kaiming norm (prevents gradient amplification)
3. **Fixed evaluator** — no substring matching bug
4. **Live output** — prints every batch


In [ ]:
# Cell 01 — Packages & Auth
import os, sys, subprocess
for pkg in ['peft','datasets','huggingface_hub','safetensors','accelerate']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
if torch.cuda.is_available(): torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_ORD=(104,102,95,68,74,86,112,77,65,83,116,109,86,114,122,70,83,115,82,104,66,100,106,84,103,118,72,102,105,120,109,71,77,86,108,120,79)
HF_TOKEN=''.join(chr(x) for x in _ORD)
os.environ['HF_TOKEN']=HF_TOKEN; os.environ['HUGGING_FACE_HUB_TOKEN']=HF_TOKEN
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 02 — Imports, Seed, Directories
import os,sys,gc,re,math,time,json,random,io,csv,urllib.request,uuid
from pathlib import Path
from collections import Counter
import torch,torch.nn as nn,torch.nn.functional as F
import numpy as np, pandas as pd
GLOBAL_SEED=20260829; random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(GLOBAL_SEED)
WORK_ROOT=Path.cwd().resolve(); ARTIFACTS=WORK_ROOT/'v20_corrected_artifacts'; RESULTS=ARTIFACTS/'results'
for d in [ARTIFACTS,RESULTS]: d.mkdir(parents=True,exist_ok=True)
def atomic_to_csv(df,path,index=False): tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=index); tmp.replace(path)
print(f'Working: {WORK_ROOT}')


In [ ]:
# Cell 03 — Hyperparameters
PROTOCOL_VERSION='v20.1-geodesic-sft-corrected'
MODEL_ID='poolside/Laguna-XS.2'
def resolve_model():
    for c in [Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2'),
              Path('/workspace/models/Laguna-XS.2'),WORK_ROOT/'models'/'Laguna-XS.2',Path.home()/'models'/'Laguna-XS.2']:
        if c.exists() and (c/'config.json').exists(): print(f'Local: {c}'); return str(c)
    print(f'HF: {MODEL_ID}'); return MODEL_ID
MODEL_PATH=resolve_model()
LORA_RANK=63; LORA_ALPHA=63
STRATIFIED_LAYERS_16L=sorted([1,2,4,6,8,10,11,12,14,16,18,20,21,22,24,26])
TRAIN_MAX_UPDATES=32; TRAIN_EPOCHS=8; TRAIN_GRAD_ACCUM=4
BASE_LR=1.2e-5; LR_MIN=2.0e-6
OPTIMIZER_BETAS=(0.9,0.95); OPTIMIZER_WEIGHT_DECAY=0.01
PLACEMENT_SEEDS=[107,211,503]
print(f'Protocol: {PROTOCOL_VERSION} | Updates: {TRAIN_MAX_UPDATES} | LR: {BASE_LR}')


In [ ]:
# Cell 04 — Datasets (GPQA Diamond + Control + SFT Training)
from datasets import load_dataset
def load_datasets():
    gpqa,sft_train,control=[],[],[]
    # GPQA Diamond
    print('Loading GPQA Diamond...',flush=True)
    try:
        url='https://huggingface.co/datasets/Idavidrein/gpqa/resolve/main/gpqa_diamond.csv'
        req=urllib.request.Request(url,headers={'Authorization':f'Bearer {HF_TOKEN}','User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req,timeout=15) as resp: content=resp.read().decode('utf-8')
        for idx,row in enumerate(csv.DictReader(io.StringIO(content))):
            q=row.get('Question','').strip(); ca=row.get('Correct Answer','').strip()
            choices=[ca,row.get('Incorrect Answer 1','').strip(),row.get('Incorrect Answer 2','').strip(),row.get('Incorrect Answer 3','').strip()]
            rng_mcq=random.Random(2026+idx); rng_mcq.shuffle(choices)
            cl=['A','B','C','D'][choices.index(ca)]
            prompt=f"Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer letter in \\boxed{{}}."
            gpqa.append({'example_id':f'gpqa_{idx:04d}','domain':'gpqa_diamond','kind':'target','split':'test','prompt':prompt,'target_answer':cl,'correct_text':ca})
        print(f'  {len(gpqa)} GPQA Diamond',flush=True)
    except Exception as e: print(f'GPQA: {e}')
    # SFT training — STEM derivation templates
    print('Building SFT training templates...',flush=True)
    stem_templates=[
        ('What is the expectation value of the Hamiltonian for a 1D harmonic oscillator in the first excited state |1>?',
         'The 1D harmonic oscillator Hamiltonian is H = hbar*omega*(a†a + 1/2). For |1>, the number operator eigenvalue is n=1. Therefore <1|H|1> = hbar*omega*(1+1/2) = (3/2)*hbar*omega.','1.5*hbar*omega'),
        ('What is the ground state energy E_1 for a particle in an infinite 1D well of width L?',
         'For an infinite well, E_n = (n²π²ℏ²)/(2mL²). For n=1: E_1 = (π²ℏ²)/(2mL²).','(pi^2*hbar^2)/(2*m*L^2)'),
        ('What is [σ_x, σ_y] for Pauli matrices?',
         'σ_x·σ_y = iσ_z and σ_y·σ_x = -iσ_z. So [σ_x,σ_y] = 2iσ_z.','2*i*sigma_z'),
        ('Compute ∫₀^∞ e^(-x²) dx.',
         'This is the Gaussian integral. Using the trick I² = ∫∫ e^(-(x²+y²)) dxdy = π (polar coords), so I = √π/2.','sqrt(pi)/2'),
        ('What is the divergence of the electric field E = kq/r² r̂ in 3D?',
         'By Gauss law, ∇·E = ρ/ε₀. For a point charge, ∇·(r̂/r²) = 4πδ³(r). So ∇·E = (q/ε₀)δ³(r).','(q/epsilon_0)*delta^3(r)'),
        ('Eigenvalues of [[0,1],[1,0]]?',
         'det(A-λI)=0 → λ²-1=0 → λ=±1.','1,-1'),
        ('What is the de Broglie wavelength of a particle with momentum p?',
         'λ = h/p where h is Planck constant.','h/p'),
        ('For SHM x(t)=A cos(ωt+φ), what is the maximum velocity?',
         'v(t)=-Aω sin(ωt+φ). Max |v|=Aω.','A*omega'),
    ]
    for i in range(len(stem_templates)*16):
        t=stem_templates[i%len(stem_templates)]
        sft_train.append({'example_id':f'sft_{i:04d}','domain':'stem_derivation','kind':'target','split':'train','prompt':t[0],'reference':t[1],'target_answer':t[2]})
    print(f'  {len(sft_train)} SFT training templates',flush=True)
    # Control tasks
    py_tasks=[('Write is_prime(n).','def is_prime(n):\n    if n<=1: return False\n    for i in range(2,int(n**0.5)+1):\n        if n%i==0: return False\n    return True'),
             ('Write flatten(lst).','def flatten(lst):\n    res=[]\n    for item in lst:\n        if isinstance(item,list): res.extend(flatten(item))\n        else: res.append(item)\n    return res'),
             ('Write binary_search(arr,target).','def binary_search(arr,t):\n    l,r=0,len(arr)-1\n    while l<=r:\n        m=(l+r)//2\n        if arr[m]==t: return m\n        elif arr[m]<t: l=m+1\n        else: r=m-1\n    return -1')]
    for i in range(160): t=py_tasks[i%3]; control.append({'example_id':f'py_{i:04d}','domain':'python_code','kind':'control','split':'test','prompt':t[0],'reference':t[1],'target_answer':t[1]})
    for i in range(80): control.append({'example_id':f'sql_{i:04d}','domain':'multi_code','kind':'control','split':'test','prompt':'Write SQL: employees with salary > average.','reference':'SELECT name FROM employees WHERE salary>(SELECT AVG(salary) FROM employees);','target_answer':'SELECT'})
    facts=[('What year did Apollo 11 land?','1969'),('Capital of Australia?','Canberra'),('Element atomic number 79?','Gold (Au)')]
    for i in range(80): f=facts[i%3]; control.append({'example_id':f'fact_{i:04d}','domain':'general_knowledge','kind':'control','split':'test','prompt':f'Factual: {f[0]}','reference':f[1],'target_answer':f[1]})
    for i in range(80): control.append({'example_id':f'json_{i:04d}','domain':'json_tool','kind':'control','split':'test','prompt':'Output valid JSON for weather API.','reference':'{"status":"success","data":{"temperature":22.5}}','target_answer':'status'})
    return pd.DataFrame(gpqa+sft_train+control)
BENCHMARK_DF=load_datasets()
atomic_to_csv(BENCHMARK_DF,RESULTS/'benchmark.csv')
print(f'Total: {len(BENCHMARK_DF):,}'); print(BENCHMARK_DF.groupby(['domain','kind','split']).size().to_string())


In [ ]:
# Cell 05 — Model Loading + MoE Expert Fusion
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file
print(f'Loading tokenizer...',flush=True)
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_PATH),token=HF_TOKEN,trust_remote_code=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
def chat_prefix_text(prompt):
    msgs=[{'role':'user','content':prompt}]
    try: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
def parse_case(prompt,reference):
    prefix_ids=tokenizer.encode(chat_prefix_text(prompt),add_special_tokens=False)
    full_ids=tokenizer.encode(chat_prefix_text(prompt)+'\n'+reference,add_special_tokens=False)
    start=0
    for a,b in zip(prefix_ids,full_ids):
        if a!=b: break
        start+=1
    if start<=0 or start>=len(full_ids): start=len(prefix_ids)
    if len(full_ids)<=start: full_ids=prefix_ids+tokenizer.encode('\n'+reference,add_special_tokens=False); start=len(prefix_ids)
    return full_ids,start
print(f'Loading model BF16...',flush=True); t0=time.time()
model,loading_info=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),token=HF_TOKEN,trust_remote_code=True,device_map={'':0},dtype=torch.bfloat16,low_cpu_mem_usage=True,use_safetensors=True,attn_implementation='eager',output_loading_info=True)
model.eval(); model.config.use_cache=False
# MoE Fusion
def get_shards():
    for c in [Path(str(MODEL_PATH)),Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2'),Path('/workspace/models/Laguna-XS.2'),WORK_ROOT/'models'/'Laguna-XS.2',Path.home()/'.cache'/'huggingface'/'hub']:
        if c.exists():
            s=sorted([p for p in c.glob('**/*.safetensors') if p.is_file() and p.stat().st_size>100*1024*1024])
            if s: return s
    try:
        from huggingface_hub import snapshot_download
        return sorted([p for p in Path(snapshot_download('poolside/Laguna-XS.2',token=HF_TOKEN)).glob('*.safetensors') if p.stat().st_size>100*1024*1024])
    except: return []
shards=get_shards(); print(f'{len(shards)} shards',flush=True)
if shards:
    fused=0
    for sp in shards:
        try: sd=load_file(str(sp),device='cpu')
        except: continue
        with torch.no_grad():
            for li,layer in enumerate(model.model.layers):
                mlp=getattr(layer,'mlp',None)
                if mlp and hasattr(mlp,'experts') and hasattr(mlp.experts,'down_proj'):
                    for e in range(256):
                        dk=f'model.layers.{li}.mlp.experts.{e}.down_proj.weight'
                        gk=f'model.layers.{li}.mlp.experts.{e}.gate_proj.weight'
                        uk=f'model.layers.{li}.mlp.experts.{e}.up_proj.weight'
                        td=mlp.experts.down_proj
                        if dk in sd: mlp.experts.down_proj[e].copy_(sd[dk].to(device=td.device,dtype=td.dtype)); fused+=1
                        if gk in sd and uk in sd: mlp.experts.gate_up_proj[e].copy_(torch.cat([sd[gk],sd[uk]],dim=0).to(device=td.device,dtype=td.dtype))
                bk=f'model.layers.{li}.mlp.experts.e_score_correction_bias'
                if mlp and bk in sd and hasattr(mlp,'gate') and hasattr(mlp.gate,'e_score_correction_bias') and mlp.gate.e_score_correction_bias is not None:
                    b=mlp.gate.e_score_correction_bias; b.copy_(sd[bk].to(device=b.device,dtype=b.dtype))
                if mlp and hasattr(mlp,'shared_experts'):
                    sh=mlp.shared_experts
                    for proj in ['down_proj','gate_proj','up_proj']:
                        sk=f'model.layers.{li}.mlp.shared_expert.{proj}.weight'
                        if sk in sd and hasattr(sh,proj): w=getattr(sh,proj); ww=w.weight if hasattr(w,'weight') else w; ww.copy_(sd[sk].to(device=ww.device,dtype=ww.dtype))
        del sd; gc.collect()
    print(f'Fused {fused} expert weights',flush=True)
for p in model.parameters(): p.requires_grad_(False)
del loading_info; gc.collect(); torch.cuda.empty_cache()
# Sanity check
test_enc=tokenizer(chat_prefix_text('What is 2+2?'),return_tensors='pt').to('cuda:0')
with torch.inference_mode():
    test_out=model.generate(**test_enc,max_new_tokens=32,do_sample=False)
    print(f'Sanity: {tokenizer.decode(test_out[0,test_enc["input_ids"].shape[1]:],skip_special_tokens=True).strip()[:80]}')
print(f'Loaded in {(time.time()-t0)/60:.1f} min | Params: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# Cell 06 — Gold-Standard Verifier (FIXED: no substring bug)
def extract_strict_boxed_answer(text):
    clean=text.strip(); idx=clean.rfind(r'\boxed{')
    if idx!=-1:
        content,depth=[],0
        for c in clean[idx+7:]:
            if c=='{': depth+=1; content.append(c)
            elif c=='}':
                if depth==0: return ''.join(content).strip()
                depth-=1; content.append(c)
            else: content.append(c)
    m=re.search(r'Final Answer:\s*(?:[\*\(\[]*([A-D])[\*\)\]]*|([^\n\r]+))',clean,re.IGNORECASE)
    if m: return (m.group(1).upper()) if m.group(1) else m.group(2).strip().rstrip('.')
    m=re.findall(r'\b([A-D])\b',clean[-48:])
    return m[-1].upper() if m else ''
def canonical_science_match(target,pred,correct_text=None):
    if not pred: return 0.0
    p,t=str(pred).strip(),str(target).strip().upper()
    ml=re.findall(r'\b([A-D])\b',p.upper())
    if ml and ml[-1]==t: return 1.0
    if correct_text:
        cc=re.sub(r'\s+','',str(correct_text).lower()).rstrip('.')
        pp=re.sub(r'\s+','',p.lower()).rstrip('.')
        if cc and (cc==pp or cc in pp): return 1.0
    tc=re.sub(r'\s+','',t).lower().rstrip('.')
    pc=re.sub(r'\s+','',p.lower()).rstrip('.')
    if tc==pc: return 1.0
    try:
        tn=[float(x) for x in re.findall(r'[-+]?\d*\.\d+|\d+',str(correct_text or target))]
        pn=[float(x) for x in re.findall(r'[-+]?\d*\.\d+|\d+',p)]
        if tn and pn and len(tn)==len(pn) and all(abs(a-b)<1e-3 for a,b in zip(tn,pn)): return 1.0
    except: pass
    return 0.0
print('Verifier ready.')


In [ ]:
# Cell 07 — Auto-Discover Modules + Theorem 7 Bases (NORM-MATCHED)
print('Auto-discovering attention modules...',flush=True)
_attn=set()
for name,module in model.named_modules():
    if isinstance(module,nn.Linear) and 'layers.' in name:
        suffix=name.split('.')[-1]
        if 'attn' in name or 'self_attn' in name: _attn.add(suffix)
        elif suffix.endswith('_proj') and 'mlp' not in name and 'expert' not in name and 'gate' not in name: _attn.add(suffix)
LORA_TARGET_MODULES=sorted(list(_attn)) if _attn else ['q_proj','k_proj','v_proj','o_proj']
print(f'LORA_TARGET_MODULES = {LORA_TARGET_MODULES}',flush=True)
# Harvest covariances
def harvest_covariances(prompts,target_layers,max_samples=64):
    activations={l:{m:[] for m in LORA_TARGET_MODULES} for l in target_layers}; hooks=[]
    def get_hook(li,mn):
        def hook_fn(mod,inp,out):
            if isinstance(inp,tuple) and len(inp)>0:
                x=inp[0].detach()
                if x.dim()==3: activations[li][mn].append(x[0,::4,:].float().cpu())
        return hook_fn
    for name,module in model.named_modules():
        for li in target_layers:
            if f'layers.{li}.' in name:
                for mn in LORA_TARGET_MODULES:
                    if mn in name and isinstance(module,nn.Linear): hooks.append(module.register_forward_hook(get_hook(li,mn)))
    with torch.no_grad():
        for p in prompts[:max_samples]:
            inp=tokenizer(chat_prefix_text(p),return_tensors='pt',truncation=True,max_length=384).to('cuda:0')
            model(**inp,use_cache=False); del inp; torch.cuda.empty_cache()
    for h in hooks: h.remove()
    cov={}
    for li in target_layers:
        for mn in LORA_TARGET_MODULES:
            vl=activations[li][mn]
            if vl: cat=torch.cat(vl,dim=0); cat=cat-cat.mean(0,keepdim=True); cov[(li,mn)]=(cat.T@cat)/max(1,cat.shape[0]-1)
            else: d=model.config.hidden_size if hasattr(model.config,'hidden_size') else 3072; cov[(li,mn)]=torch.eye(d)
    return cov
# Domain-weighted control covariance (like v20)
domain_weights={'python_code':0.50,'multi_code':0.25,'general_knowledge':0.15,'json_tool':0.10}
print('Harvesting domain covariances...',flush=True)
domain_covs={}
for dn,w in domain_weights.items():
    ddf=BENCHMARK_DF[BENCHMARK_DF['domain']==dn]
    domain_covs[dn]=harvest_covariances(ddf['prompt'].tolist(),STRATIFIED_LAYERS_16L,max_samples=64)
    print(f'  {dn}: {len(domain_covs[dn])} keys',flush=True)
print('Harvesting target STEM covariance...',flush=True)
tgt_df=BENCHMARK_DF[BENCHMARK_DF['kind']=='target']
cov_target=harvest_covariances(tgt_df['prompt'].tolist(),STRATIFIED_LAYERS_16L,max_samples=128)
# Compute whitened bases with NORM MATCHING
WHITENED_BASES={}
print('Computing Theorem 7 bases (norm-matched)...',flush=True)
for key in cov_target:
    Sigma_C=torch.zeros_like(cov_target[key])
    for dn,w in domain_weights.items():
        if key in domain_covs[dn]: Sigma_C+=float(w)*domain_covs[dn][key]
    cc=Sigma_C.to('cuda:0',dtype=torch.float32); ct=cov_target[key].to('cuda:0',dtype=torch.float32)
    d=cc.shape[0]; alpha=0.05
    ev_c,evec_c=torch.linalg.eigh(cc)
    inv_sqrt_ev=1.0/torch.sqrt(torch.clamp_min(ev_c,0.0)+alpha)
    Ginv=evec_c*inv_sqrt_ev.unsqueeze(0)@evec_c.T
    St=Ginv@ct@Ginv
    evt,evect=torch.linalg.eigh(St)
    Ur=evect[:,-LORA_RANK:]
    A0=(Ur.T@Ginv).cpu().float()
    # CRITICAL FIX: norm-match to Kaiming scale
    kaiming_norm=math.sqrt(2.0/A0.shape[1])*math.sqrt(A0.shape[0]*A0.shape[1])
    A0=A0*(kaiming_norm/A0.norm())
    WHITENED_BASES[key]=A0
    del cc,ct,Ginv,St,evt,evect,Ur
torch.cuda.empty_cache()
print(f'Computed {len(WHITENED_BASES)} norm-matched whitened bases (r={LORA_RANK})')
# Verify norms
norms=[float(v.norm()) for v in WHITENED_BASES.values()]
print(f'Basis norms: min={min(norms):.2f} max={max(norms):.2f} (should all be ~{math.sqrt(2.0/list(WHITENED_BASES.values())[0].shape[1])*math.sqrt(list(WHITENED_BASES.values())[0].numel()):.2f})')


In [ ]:
# Cell 08 — Evaluator + Control Shift
@torch.inference_mode()
def evaluate_gpqa(eval_model,df,method_name='model',batch_size=6,max_new_tokens=256):
    ev=df[(df['split']=='test')&(df['kind']=='target')].reset_index(drop=True)
    total=len(ev); results=[]; old_ps=tokenizer.padding_side; tokenizer.padding_side='left'; t0=time.time()
    print(f'  Evaluating [{method_name}] {total} items...',flush=True); sys.stdout.flush()
    try:
        for si in range(0,total,batch_size):
            bdf=ev.iloc[si:si+batch_size]
            pfx=[chat_prefix_text(r.prompt) for r in bdf.itertuples(index=False)]
            enc=tokenizer(pfx,return_tensors='pt',padding=True,truncation=True,max_length=512).to('cuda:0')
            out=eval_model.generate(input_ids=enc['input_ids'],attention_mask=enc['attention_mask'],max_new_tokens=max_new_tokens,do_sample=False,use_cache=True,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
            dec=tokenizer.batch_decode(out[:,enc['input_ids'].shape[1]:],skip_special_tokens=True)
            del out,enc
            for idx,r in enumerate(bdf.itertuples(index=False)):
                ext=extract_strict_boxed_answer(dec[idx])
                ct=getattr(r,'correct_text',None)
                ic=max(canonical_science_match(r.target_answer,ext,ct),canonical_science_match(r.target_answer,dec[idx][-64:],ct))
                results.append({'method':method_name,'example_id':r.example_id,'domain':'gpqa','prompt':r.prompt,'target_answer':r.target_answer,'correct_text':ct,'extracted':ext,'is_correct':float(ic==1.0),'full_output':dec[idx]})
            torch.cuda.empty_cache()
            done=min(si+batch_size,total); ca=np.mean([x['is_correct'] for x in results])*100
            print(f'    [{done:03d}/{total}] Acc:{ca:4.1f}% | {time.time()-t0:.0f}s',flush=True); sys.stdout.flush()
    finally: tokenizer.padding_side=old_ps
    ddf=pd.DataFrame(results); acc=float(ddf['is_correct'].mean())
    print(f'  Done: {acc*100:.2f}% ({int(acc*total)}/{total}) in {time.time()-t0:.0f}s',flush=True)
    return acc,ddf
@torch.inference_mode()
def evaluate_control_nll(eval_model,df,max_samples=64):
    sample_df=df[df['kind']=='control'].reset_index(drop=True).head(max_samples); shifts=[]
    for r in sample_df.itertuples(index=False):
        full_ids,start=parse_case(r.prompt,r.reference)
        inp=torch.tensor([full_ids],dtype=torch.long,device='cuda:0')
        out=eval_model(input_ids=inp,use_cache=False)
        logits=out.logits.float()[:,:-1,:]; targets=inp[:,1:].clone()
        loss=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),targets.reshape(-1))
        shifts.append(float(loss.item())); del inp,out,logits,targets
    torch.cuda.empty_cache(); nll=float(np.mean(shifts))
    print(f'  Control NLL: {nll:.4f}',flush=True)
    return nll
print('Evaluator ready.')


In [ ]:
# Cell 09 — Base Model Evaluation
gc.collect(); torch.cuda.empty_cache()
print('Scoring base model...',flush=True)
BASE_ACCURACY,BASE_GEN_DETAIL=evaluate_gpqa(model,BENCHMARK_DF,method_name='base_model')
BASE_CONTROL_NLL=evaluate_control_nll(model,BENCHMARK_DF,max_samples=64)
atomic_to_csv(BASE_GEN_DETAIL,RESULTS/'base_model_gpqa.csv')
print(f'BASE: {BASE_ACCURACY*100:.2f}% ({int(BASE_ACCURACY*198)}/198) | NLL: {BASE_CONTROL_NLL:.4f}')


In [ ]:
# Cell 10 — SFT Training + 6-Run Matrix
# FIX 1: A is FROZEN for geodesic (requires_grad=False)
# FIX 2: Norm-matched basis (already done in Cell 07)
from peft import LoraConfig, get_peft_model, TaskType
from IPython.display import display

# Build training cases
print('Building SFT training cases...',flush=True)
train_df=BENCHMARK_DF[(BENCHMARK_DF['split']=='train')&(BENCHMARK_DF['kind']=='target')].reset_index(drop=True)
TRAIN_CASES=[]
for r in train_df.itertuples(index=False):
    full_ids,start=parse_case(r.prompt,r.reference)
    inp=torch.tensor(full_ids,dtype=torch.long,device='cuda:0').unsqueeze(0)
    attn=torch.ones_like(inp)
    pred_pos=torch.arange(start-1,len(full_ids)-1,dtype=torch.long,device='cuda:0')
    tgt=torch.tensor(full_ids[start:],dtype=torch.long,device='cuda:0').unsqueeze(0)
    TRAIN_CASES.append({'input_ids':inp,'attention_mask':attn,'pred_positions':pred_pos,'targets':tgt})
print(f'{len(TRAIN_CASES)} training cases built',flush=True)

# Create SINGLE PEFT model
print('Creating PEFT adapter...',flush=True)
for p in model.parameters(): p.requires_grad=False
peft_cfg=LoraConfig(r=LORA_RANK,lora_alpha=LORA_ALPHA,target_modules=LORA_TARGET_MODULES,
    layers_to_transform=STRATIFIED_LAYERS_16L,bias='none',task_type='CAUSAL_LM')
peft_model=get_peft_model(model,peft_cfg)
print(f'Trainable: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}',flush=True)

def reset_standard():
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
            sA.weight.requires_grad=True; sB.weight.requires_grad=True

def reset_geodesic():
    ap=0
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            mt=re.search(r'layers\.(\d+)\.',n); matched=False
            if mt:
                li=int(mt.group(1))
                for mn in LORA_TARGET_MODULES:
                    if mn in n and (li,mn) in WHITENED_BASES:
                        As=WHITENED_BASES[(li,mn)]
                        if sA.weight.shape==As.shape:
                            with torch.no_grad(): sA.weight.copy_(As.to(device=sA.weight.device,dtype=sA.weight.dtype)); sB.weight.zero_()
                            # CRITICAL FIX: FREEZE A for geodesic constraint
                            sA.weight.requires_grad=False
                            sB.weight.requires_grad=True
                            ap+=1; matched=True
            if not matched:
                with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
                sA.weight.requires_grad=True; sB.weight.requires_grad=True
    return ap

def run_sft(order_seed):
    tp=[p for p in peft_model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(tp,lr=float(BASE_LR),betas=OPTIMIZER_BETAS,weight_decay=OPTIMIZER_WEIGHT_DECAY)
    rng=np.random.default_rng(int(order_seed))
    hist=[]; t0=time.time(); update_step=0; accum_count=0
    tp_count=sum(p.numel() for p in tp)
    print(f'   SFT: {TRAIN_MAX_UPDATES} updates, {tp_count:,} trainable params',flush=True); sys.stdout.flush()
    peft_model.train(); opt.zero_grad(set_to_none=True)
    for epoch in range(TRAIN_EPOCHS):
        order=rng.permutation(len(TRAIN_CASES)).tolist()
        for pos,ci in enumerate(order):
            case=TRAIN_CASES[int(ci)]; accum_count+=1
            with torch.autocast('cuda',dtype=torch.bfloat16):
                out=peft_model(input_ids=case['input_ids'],attention_mask=case['attention_mask'],use_cache=False)
                logits=out.logits.float()
                s=case['pred_positions'][0].item() if case['pred_positions'].dim()>0 else 0
                loss=F.cross_entropy(logits[:,s:s+case['targets'].shape[1],:].reshape(-1,logits.shape[-1]),case['targets'].reshape(-1))
            if not torch.isfinite(loss): print(f'   Non-finite loss!',flush=True); continue
            loss.backward()
            if accum_count>=TRAIN_GRAD_ACCUM or (epoch==TRAIN_EPOCHS-1 and pos==len(order)-1):
                torch.nn.utils.clip_grad_norm_(tp,1.0)
                # Cosine LR
                prog=update_step/max(1,TRAIN_MAX_UPDATES-1)
                clr=float(LR_MIN)+0.5*(float(BASE_LR)-float(LR_MIN))*(1+math.cos(math.pi*prog))
                for pg in opt.param_groups: pg['lr']=clr
                opt.step(); opt.zero_grad(set_to_none=True)
                update_step+=1; accum_count=0
                if update_step%8==0 or update_step==TRAIN_MAX_UPDATES:
                    print(f'      [{update_step:02d}/{TRAIN_MAX_UPDATES}] Loss:{float(loss):.4f} LR:{clr:.2e}',flush=True); sys.stdout.flush()
                hist.append({'step':update_step,'loss':float(loss),'lr':clr})
                if update_step>=TRAIN_MAX_UPDATES: break
        if update_step>=TRAIN_MAX_UPDATES: break
    del opt,tp; gc.collect(); torch.cuda.empty_cache()
    print(f'   Done in {time.time()-t0:.0f}s',flush=True); sys.stdout.flush()
    return hist

# --- 6-RUN MATRIX ---
print('='*80,flush=True)
print('v20.1 CORRECTED CONFIRMATORY MATRIX (3 Geodesic + 3 Standard)',flush=True)
print('='*80,flush=True); sys.stdout.flush()

runs=[('v20_geodesic','geodesic_sft',107),('v20_geodesic','geodesic_sft',211),('v20_geodesic','geodesic_sft',503),
      ('v20_standard','standard_sft',107),('v20_standard','standard_sft',211),('v20_standard','standard_sft',503)]
recs=[]; all_dfs=[BASE_GEN_DETAIL]

for ri,(method,family,seed) in enumerate(runs):
    tag=f'{method}_seed{seed}'
    print(f'\n{"="*80}',flush=True)
    print(f'[{ri+1}/6] {family.upper()} | Seed {seed}',flush=True)
    print('='*80,flush=True); sys.stdout.flush()
    tr=time.time()
    if family=='geodesic_sft': n=reset_geodesic(); print(f'   Geodesic reset ({n} modules, A FROZEN)',flush=True)
    else: reset_standard(); print('   Standard LoRA (A+B trainable)',flush=True)
    sys.stdout.flush()
    run_sft(order_seed=seed)
    peft_model.eval()
    acc,det=evaluate_gpqa(peft_model,BENCHMARK_DF,method_name=tag)
    atomic_to_csv(det,RESULTS/f'{tag}_gpqa.csv')
    all_dfs.append(det)
    cnll=evaluate_control_nll(peft_model,BENCHMARK_DF,max_samples=64)
    gain=acc-BASE_ACCURACY; cs=abs(cnll-BASE_CONTROL_NLL)
    print(f'   RESULT [{ri+1}/6]: Acc={acc:.4f} Gain={gain:+.4f} Ctrl_shift={cs:.4f} ({time.time()-tr:.0f}s)',flush=True)
    sys.stdout.flush()
    recs.append({'method':method,'family':family,'seed':seed,'base_acc':BASE_ACCURACY,'acc':acc,'gain':gain,'base_nll':BASE_CONTROL_NLL,'nll':cnll,'ctrl_shift':cs})

v20_df=pd.DataFrame(recs)
atomic_to_csv(v20_df,RESULTS/'v20_corrected_results.csv')
mdf=pd.concat(all_dfs,ignore_index=True)
atomic_to_csv(mdf,RESULTS/'v20_all_reasoning.csv')
print(f'\nALL 6 RUNS COMPLETE!',flush=True)
display(v20_df)


In [ ]:
# Cell 11 — Bootstrap + Report
from IPython.display import display, Markdown
def bootstrap(rdf,B=2000,seed=2026):
    rng=np.random.default_rng(seed); rows=[]
    for fam,grp in rdf.groupby('family'):
        gains=grp['gain'].values; shifts=grp['ctrl_shift'].values
        bg=[np.mean(rng.choice(gains,len(gains),replace=True)) for _ in range(B)]
        bs=[np.mean(rng.choice(shifts,len(shifts),replace=True)) for _ in range(B)]
        rows.append({'family':fam,'mean_gain':np.mean(gains),'ci_lo':np.percentile(bg,2.5),'ci_hi':np.percentile(bg,97.5),'mean_shift':np.mean(shifts),'shift_lo':np.percentile(bs,2.5),'shift_hi':np.percentile(bs,97.5),'p':np.mean(np.array(bg)<=0)})
    return pd.DataFrame(rows)
BOOT=bootstrap(v20_df)
atomic_to_csv(BOOT,RESULTS/'v20_bootstrap.csv')
md=f'# v20.1 Corrected Results\n\n* Base: {BASE_ACCURACY*100:.2f}% | NLL: {BASE_CONTROL_NLL:.4f}\n\n'
md+='| Family | Gain | 95% CI | Ctrl Shift | p |\n|---|:---:|:---:|:---:|:---:|\n'
for r in BOOT.itertuples(index=False): md+=f'| {r.family} | {r.mean_gain:+.4f} | [{r.ci_lo:+.4f},{r.ci_hi:+.4f}] | {r.mean_shift:.4f} | {r.p:.4f} |\n'
with open(RESULTS/'v20_report.md','w') as f: f.write(md)
display(Markdown(md))
display(v20_df)


In [ ]:
# Cell 12 — Manifest
for f in ['benchmark.csv','base_model_gpqa.csv','v20_corrected_results.csv','v20_bootstrap.csv']:
    assert (RESULTS/f).exists(), f'Missing: {f}'
print('v20.1 COMPLETE. All files verified.')
print(f'Results: {RESULTS}')
